# IPCC LULUCF zonal stats → country summary (area comparison + fluxes)

Reproduces, for every country in the EU27 zonal-stats workbook, the summaries that were built by hand in the
Sweden `KEEP` workbook (`area_comparison` tab and the top block of the `combined` / `fluxes_combined` tab).

**Inputs**

1. `IPCC_LULUCF_zonal_stats_summary__EU27__*.xlsx` — tabs `flux` and `ipcc_class_area`, written by
   `1_adm0_LULUCF_vegetation_IPCC_summary_spreadsheet.ipynb` **after** the `ipcc_rules.csv` reassignment (the newest workbook in `ZONAL_STATS_DIR` is used)
2. `EU27_CRT41_land_use_area.xlsx` (`CRT_XLSX`) — CRT Table 4.1 land-use transition matrices (kha) scraped from the national inventory submissions

**Outputs** (one workbook, `*_country_summary.xlsx`)

| tab | content |
|---|---|
| `area_comparison` | 2024 area (Mha) per IPCC class: WRI vs CRT4.1 (managed land only), difference, notes |
| `fluxes_CO2` | mean annual flux 2016–2024, CO2 only (Mt CO2 / yr) by category × pool |
| `fluxes_CO2e` | mean annual flux 2016–2024, all gases (Mt CO2e / yr) by category × pool |
| `flux_matrix` | the underlying from→to transition matrices (long format) for traceability |

**Category definitions** (as in the Sweden `combined` tab — rows of the pivot are the *initial* class `IPCC_initial`,
columns are the *current* class `IPCC_class`):

* `FOREST` – everything whose current class is Forest land (Forest remaining Forest **and** conversions *to* Forest)
* `DEFORESTATION` – initial class Forest land, current class anything other than Forest land
* `OTHER` – all remaining transitions (non-forest → non-forest)
* `LULUCF` – FOREST + DEFORESTATION + OTHER

`total = vegetation + mineral soils + organic soils`, where organic soils = drained + burned.


## 1. Configuration

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- input files -----------------------------------------------------------
# Folder notebook 1 writes to (output_dir); the newest IPCC_LULUCF_zonal_stats_summary__EU27__*.xlsx in it is used
ZONAL_STATS_DIR  = Path(r"/mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/reassigned")
_workbooks = sorted(ZONAL_STATS_DIR.glob("IPCC_LULUCF_zonal_stats_summary__EU27__*.xlsx"), key=lambda p: p.stat().st_mtime)
if not _workbooks:
    raise FileNotFoundError(f"No IPCC_LULUCF_zonal_stats_summary__EU27__*.xlsx in {ZONAL_STATS_DIR}. "
                            "Run 1_adm0_LULUCF_vegetation_IPCC_summary_spreadsheet.ipynb first (it writes the workbook there), "
                            "or set ZONAL_STATS_DIR to the folder that has it.")
ZONAL_STATS_XLSX = _workbooks[-1]
CRT_XLSX         = Path(r"/mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27_CRT41_land_use_area.xlsx")   # CRT Table 4.1 areas
assert CRT_XLSX.exists(), f"CRT4.1 workbook not found: {CRT_XLSX}"
print("zonal stats workbook:", ZONAL_STATS_XLSX.name)

# ---- output (next to the input workbook, same timestamp suffix) -------------
OUT_XLSX = ZONAL_STATS_DIR / ZONAL_STATS_XLSX.name.replace("IPCC_LULUCF_zonal_stats_summary__", "IPCC_LULUCF_country_summary__")

# ---- analysis settings -----------------------------------------------------
AREA_YEAR      = 2024                 # year used in area_comparison (both WRI and CRT4.1)
CRT_FALLBACK_TO_LATEST_YEAR = True    # if a country has no CRT4.1 submission for AREA_YEAR, use its latest year and flag it in Notes
FLUX_YEARS     = None                 # None -> use all years present in the flux tab (2016-2024); or e.g. range(2016, 2025)

# Optional country-specific exclusions of transitions, mirroring the Sweden sheet where
# "Wetlands remaining Wetlands" (IPCC_change 55) was zeroed because all Swedish wetland is unmanaged.
# Keys are adm0 codes, values are lists of IPCC_change codes to drop from the flux summary.
# Leave empty to keep everything.
EXCLUDE_TRANSITIONS = {
    # "SWE": [55],
}

# ---- class mappings --------------------------------------------------------
# WRI IPCC_class_name -> summary class (Unassigned dropped; Bare / Water / Snow-Ice -> Other)
WRI_CLASS_MAP = {
    "Settlements":            "Settlements",
    "Cropland":               "Cropland",
    "Forest Land":            "Forest land",
    "Grassland":              "Grassland",
    "Wetlands":               "Wetlands",
    "Other Land (Bare)":      "Other",
    "Other Land (Water)":     "Other",
    "Other Land (Snow/Ice)":  "Other",
    # "Unassigned" intentionally absent -> dropped
}

# CRT4.1 "To" (current-year) categories -> summary class. Only MANAGED land is kept;
# unmanaged categories and the "Total unmanaged land" line are dropped.
CRT_CLASS_MAP = {
    "Settlements":            "Settlements",
    "Cropland":               "Cropland",
    "Forest land (managed)":  "Forest land",
    "Grassland (managed)":    "Grassland",
    "Wetlands (managed)":     "Wetlands",
    "Other land":             "Other",
    # dropped: "Forest land (unmanaged)", "Grassland (unmanaged)", "Wetlands (unmanaged)", "Total unmanaged land"
}

CLASS_ORDER = ["Settlements", "Cropland", "Forest land", "Grassland", "Wetlands", "Other"]

# ---- flux layers per gas basis and pool ------------------------------------
# Vegetation layers have no veg_ prefix in current zonal stats outputs; older workbooks with veg_ names also work (see section 2)
FLUX_LAYERS = {
    "CO2": {
        "vegetation":    ["net_flux__all_C_pools__CO2_only__MgCO2"],
        "mineral soils": ["SOC_net__mineral_soil_extent__0-30cm_MgCO2"],
        "organic soils": ["organic_soil_drained_co2_Mg_CO2", "organic_soil_burned_co2_Mg_CO2"],
    },
    "CO2e": {
        "vegetation":    ["net_flux__all_C_pools__all_gases__MgCO2e"],
        "mineral soils": ["SOC_net__mineral_soil_extent__0-30cm_MgCO2"],   # SOC is CO2 only; same layer on both bases
        "organic soils": ["organic_soil_drained_total_Mg_CO2e", "organic_soil_burned_total_Mg_CO2e"],
    },
}
POOLS = ["vegetation", "mineral soils", "organic soils"]
CATEGORY_ORDER = ["LULUCF", "FOREST", "DEFORESTATION", "OTHER"]
FOREST_CODE = 3   # IPCC_class / IPCC_initial code for Forest land


zonal stats workbook: IPCC_LULUCF_zonal_stats_summary__EU27__EU27__20261021_01_57_57.xlsx


## 2. Load inputs

In [3]:
flux = pd.read_excel(ZONAL_STATS_XLSX, sheet_name="flux")
area = pd.read_excel(ZONAL_STATS_XLSX, sheet_name="ipcc_class_area")
crt  = pd.read_excel(CRT_XLSX, usecols=["Country", "Year", "From", "To", "Area (kha)"])

countries = (flux[["adm0", "country_name"]].drop_duplicates()
             .sort_values("adm0").reset_index(drop=True))
print(f"flux: {flux.shape}, area: {area.shape}, crt: {crt.shape}")
print(f"{len(countries)} countries: {', '.join(countries.adm0)}")

# Vegetation layer names: current outputs have no veg_ prefix (net_flux__...), older ones do (veg_net_flux__...).
# Uses whichever name the flux tab has.
available_layers = set(flux.analysis_layer.unique())
for basis, pools in FLUX_LAYERS.items():
    resolved = []
    for l in pools["vegetation"]:
        alt = l[len("veg_"):] if l.startswith("veg_") else f"veg_{l}"
        resolved.append(alt if (l not in available_layers and alt in available_layers) else l)
    pools["vegetation"] = resolved
print("vegetation layers:", {basis: pools["vegetation"] for basis, pools in FLUX_LAYERS.items()})

# sanity: every layer we need exists
missing = {l for pools in FLUX_LAYERS.values() for ls in pools.values() for l in ls} - set(flux.analysis_layer.unique())
assert not missing, f"layers missing from flux tab: {missing}"
unmapped = set(area.IPCC_class_name.unique()) - set(WRI_CLASS_MAP) - {"Unassigned"}
assert not unmapped, f"unmapped WRI classes: {unmapped}"


flux: (94209, 16), area: (10823, 11), crt: (2188, 5)
27 countries: AUT, BEL, BGR, CYP, CZE, DEU, DNK, ESP, EST, FIN, FRA, GRC, HRV, HUN, IRL, ITA, LTU, LUX, LVA, MLT, NLD, POL, PRT, ROU, SVK, SVN, SWE
vegetation layers: {'CO2': ['net_flux__all_C_pools__CO2_only__MgCO2'], 'CO2e': ['net_flux__all_C_pools__all_gases__MgCO2e']}


## 3. Area comparison (WRI vs CRT4.1, Mha)

* WRI: `ipcc_class_area`, `year == AREA_YEAR`, summed by `IPCC_class_name` after remapping; Unassigned dropped.
* CRT4.1: Table 4.1 transition matrix, `Year == AREA_YEAR`; the area of a class is the **column total of the "To" (end-of-year) class**
  summed over all "From" rows (that is what `SUM(Sweden!$T$30:$T$35)` etc. did in the Sweden sheet). Only managed categories are kept.
  Notation keys (`NO`, `NE`, `IE`, `NA`, text) are treated as 0 — see the note column.


In [4]:
# ---- WRI ---------------------------------------------------------------------
wri = area[area.year == AREA_YEAR].copy()
wri["summary_class"] = wri.IPCC_class_name.map(WRI_CLASS_MAP)
wri = wri.dropna(subset=["summary_class"])
wri = (wri.groupby(["adm0", "country_name", "summary_class"], as_index=False).area_ha.sum()
          .assign(WRI_Mha=lambda d: d.area_ha / 1e6)
          .drop(columns="area_ha"))

# ---- CRT4.1 ------------------------------------------------------------------
crt = crt.copy()
crt["area_kha_num"] = pd.to_numeric(crt["Area (kha)"], errors="coerce")
crt["is_notation_key"] = crt["Area (kha)"].notna() & crt.area_kha_num.isna()   # NO / NE / IE / free text
crt["area_kha_num"] = crt.area_kha_num.fillna(0.0)

# pick the year per country (AREA_YEAR, optionally falling back to the latest available)
crt_year = crt.groupby("Country").Year.agg(lambda s: AREA_YEAR if AREA_YEAR in s.values else s.max()).rename("crt_year")
crt = crt.join(crt_year, on="Country")
crt = crt[crt.Year == crt.crt_year]
if not CRT_FALLBACK_TO_LATEST_YEAR:
    crt = crt[crt.Year == AREA_YEAR]

# keep only rows from real classes (drop the 'Total unmanaged land' From-line) and managed To-classes
crt = crt[crt.From != "Total unmanaged land"]
crt["summary_class"] = crt.To.map(CRT_CLASS_MAP)
crt_m = crt.dropna(subset=["summary_class"])

crt_sum = (crt_m.groupby(["Country", "crt_year", "summary_class"], as_index=False)
                .agg(CRT_Mha=("area_kha_num", lambda s: s.sum() / 1e3),
                     n_notation_keys=("is_notation_key", "sum")))

# ---- combine -----------------------------------------------------------------
grid = (countries.merge(pd.DataFrame({"summary_class": CLASS_ORDER}), how="cross"))
ac = (grid.merge(wri, on=["adm0", "country_name", "summary_class"], how="left")
          .merge(crt_sum, left_on=["country_name", "summary_class"], right_on=["Country", "summary_class"], how="left")
          .drop(columns=["Country"]))
ac["WRI_Mha"] = ac.WRI_Mha.fillna(0.0)
ac["difference_Mha"] = ac.WRI_Mha - ac.CRT_Mha

def _note(r):
    notes = []
    if pd.isna(r.CRT_Mha):
        notes.append("No CRT4.1 data")
    elif r.crt_year != AREA_YEAR:
        notes.append(f"CRT4.1 from {int(r.crt_year)} (no {AREA_YEAR} submission)")
    if r.CRT_Mha == 0 and r.summary_class in ("Wetlands", "Grassland", "Forest land"):
        notes.append(f"No managed {r.summary_class} reported in CRT4.1")
    return "; ".join(notes)
ac["Notes"] = ac.apply(_note, axis=1)

# per-country total row
tot = (ac.groupby(["adm0", "country_name"], as_index=False)[["WRI_Mha", "CRT_Mha"]].sum(min_count=1)
         .assign(summary_class="Total"))
tot["difference_Mha"] = tot.WRI_Mha - tot.CRT_Mha
tot["crt_year"] = ac.groupby(["adm0", "country_name"]).crt_year.first().values
tot["Notes"] = ""

area_comparison = (pd.concat([ac, tot], ignore_index=True)
                     .rename(columns={"summary_class": "IPCC_class_name", "crt_year": "CRT_year"}))
order = {c: i for i, c in enumerate(CLASS_ORDER + ["Total"])}
area_comparison = (area_comparison.assign(_o=area_comparison.IPCC_class_name.map(order))
                     .sort_values(["adm0", "_o"]).drop(columns=["_o", "n_notation_keys"])
                     [["adm0", "country_name", "IPCC_class_name", "WRI_Mha", "CRT_Mha", "difference_Mha", "CRT_year", "Notes"]]
                     .reset_index(drop=True))
area_comparison.head(14)


,adm0,country_name,IPCC_class_name,WRI_Mha,CRT_Mha,difference_Mha,CRT_year,Notes
0,AUT,Austria,Settlements,2.515351,0.571711,1.943640,2024,
1,AUT,Austria,Cropland,0.955701,1.392281,-0.436581,2024,
2,AUT,Austria,Forest land,3.208451,4.026405,-0.817954,2024,
3,AUT,Austria,Grassland,1.446083,1.575756,-0.129673,2024,
4,AUT,Austria,Wetlands,0.081427,0.156077,-0.074650,2024,
5,AUT,Austria,Other,0.174442,0.665769,-0.491327,2024,
6,AUT,Austria,Total,8.381457,8.388000,-0.006543,2024,
7,BEL,Belgium,Settlements,1.495329,0.701384,0.793945,2023,CRT4.1 from 2023 (no 2024 submission)
8,BEL,Belgium,Cropland,0.620446,0.900574,-0.280128,2023,CRT4.1 from 2023 (no 2024 submission)
9,BEL,Belgium,Forest land,0.610818,0.719419,-0.108600,2023,CRT4.1 from 2023 (no 2024 submission)


## 4. Fluxes by category (mean annual, Mt / yr)

For each country, gas basis and pool, build the from→to matrix (`IPCC_initial` × `IPCC_class`) of the summed flux
over all years, divide by the number of years and by 10<sup>6</sup> to get Mt / yr, then collapse into the four categories.
Unassigned (code 0) on either axis is dropped, as in the pivot tables of the Sweden sheet.


In [5]:
years = sorted(flux.year.unique()) if FLUX_YEARS is None else sorted(FLUX_YEARS)
n_years = len(years)
print(f"averaging over {n_years} years: {years[0]}-{years[-1]}")

f = flux[flux.year.isin(years) & (flux.IPCC_initial != 0) & (flux.IPCC_class != 0)].copy()

# optional country-specific exclusions
for a0, codes in EXCLUDE_TRANSITIONS.items():
    f = f[~((f.adm0 == a0) & f.IPCC_change.isin(codes))]

# long table: adm0, gas_basis, pool, from, to -> Mt/yr
pieces = []
for basis, pools in FLUX_LAYERS.items():
    for pool, layers in pools.items():
        sub = f[f.analysis_layer.isin(layers)]
        g = (sub.groupby(["adm0", "country_name", "IPCC_initial", "IPCC_class"], as_index=False).value.sum())
        g["Mt_per_yr"] = g.value / n_years / 1e6
        g["gas_basis"] = basis
        g["pool"] = pool
        pieces.append(g.drop(columns="value"))
flux_matrix = pd.concat(pieces, ignore_index=True)

def categorize(df):
    is_to_forest   = df.IPCC_class == FOREST_CODE
    is_from_forest = df.IPCC_initial == FOREST_CODE
    cat = np.select([is_to_forest, is_from_forest & ~is_to_forest], ["FOREST", "DEFORESTATION"], default="OTHER")
    return cat
flux_matrix["category"] = categorize(flux_matrix)

def summarise(basis):
    m = flux_matrix[flux_matrix.gas_basis == basis]
    by_cat = (m.groupby(["adm0", "country_name", "category", "pool"]).Mt_per_yr.sum()
                .unstack("pool").reindex(columns=POOLS).fillna(0.0))
    lulucf = by_cat.groupby(level=["adm0", "country_name"]).sum()
    lulucf["category"] = "LULUCF"
    lulucf = lulucf.set_index("category", append=True)
    out = pd.concat([by_cat, lulucf]).reset_index()
    # make sure every country has all four categories
    grid = countries.merge(pd.DataFrame({"category": CATEGORY_ORDER}), how="cross")
    out = grid.merge(out, on=["adm0", "country_name", "category"], how="left").fillna(0.0)
    out.insert(3, "total", out[POOLS].sum(axis=1))
    return out[["adm0", "country_name", "category", "total"] + POOLS]

fluxes_CO2  = summarise("CO2")
fluxes_CO2e = summarise("CO2e")
fluxes_CO2e[fluxes_CO2e.adm0 == "SWE"]


averaging over 9 years: 2016-2024


,adm0,country_name,category,total,vegetation,mineral soils,organic soils
104,SWE,Sweden,LULUCF,207.665477,-38.262422,6.799490,239.128410
105,SWE,Sweden,FOREST,77.285412,-25.169178,-0.353640,102.808230
106,SWE,Sweden,DEFORESTATION,4.658151,2.538488,-0.001112,2.120775
107,SWE,Sweden,OTHER,125.721915,-15.631732,7.154243,134.199405


## 5. Write the output workbook

In [6]:
class_names = {1: "Settlements", 2: "Cropland", 3: "Forest land", 4: "Grassland", 5: "Wetlands",
               6: "Other land (Bare)", 7: "Other land (Water)", 8: "Other land (Snow/Ice)"}
flux_matrix_out = (flux_matrix.assign(from_class=flux_matrix.IPCC_initial.map(class_names),
                                      to_class=flux_matrix.IPCC_class.map(class_names))
                   [["adm0", "country_name", "gas_basis", "pool", "category",
                     "IPCC_initial", "from_class", "IPCC_class", "to_class", "Mt_per_yr"]]
                   .sort_values(["adm0", "gas_basis", "pool", "IPCC_initial", "IPCC_class"]))

readme = pd.DataFrame({"item": [
    "input workbook", "area_comparison", "fluxes_CO2", "fluxes_CO2e", "flux_matrix",
    "FOREST", "DEFORESTATION", "OTHER", "LULUCF", "total",
    "flux averaging", "CRT4.1 managed classes", "CRT4.1 notation keys", "WRI 'Other'",
], "description": [
    f"{ZONAL_STATS_XLSX.name} (IPCC classes after ipcc_rules.csv reassignment in notebook 1)",
    f"Area in {AREA_YEAR} (Mha). WRI = ipcc_class_area tab; CRT4.1 = column ('To') totals of Table 4.1, managed land only. difference = WRI - CRT4.1",
    "Mean annual net flux, CO2 only (Mt CO2 / yr)",
    "Mean annual net flux, all gases (Mt CO2e / yr)",
    "Underlying from->to matrices used to build the flux tabs (Mt / yr)",
    "current class = Forest land (forest remaining forest + all conversions to forest)",
    "initial class = Forest land, current class != Forest land",
    "all other transitions (non-forest -> non-forest)",
    "FOREST + DEFORESTATION + OTHER",
    "vegetation + mineral soils + organic soils (drained + burned)",
    f"sum over {years[0]}-{years[-1]} divided by {n_years} years; Unassigned (code 0) dropped; exclusions: {EXCLUDE_TRANSITIONS or 'none'}",
    ", ".join(CRT_CLASS_MAP),
    "NO / NE / IE / NA / text treated as 0",
    "Other Land (Bare) + Other Land (Water) + Other Land (Snow/Ice); Unassigned dropped",
]})

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    readme.to_excel(xw, sheet_name="README", index=False)
    area_comparison.to_excel(xw, sheet_name="area_comparison", index=False)
    fluxes_CO2.to_excel(xw, sheet_name="fluxes_CO2", index=False)
    fluxes_CO2e.to_excel(xw, sheet_name="fluxes_CO2e", index=False)
    flux_matrix_out.to_excel(xw, sheet_name="flux_matrix", index=False)

    # light formatting: column widths + number formats
    for ws in xw.book.worksheets:
        for col in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in col[:200]) + 2
            ws.column_dimensions[col[0].column_letter].width = min(max(width, 8), 60)
        ws.freeze_panes = "A2"
    for name in ["area_comparison", "fluxes_CO2", "fluxes_CO2e", "flux_matrix"]:
        ws = xw.book[name]
        for row in ws.iter_rows(min_row=2):
            for c in row:
                if isinstance(c.value, float):
                    c.number_format = "0.000"
print(f"written {OUT_XLSX.resolve()}")


written /mnt/c/gis/afolu_model/land_use/zonal_statistics/EU27/reassigned/IPCC_LULUCF_country_summary__EU27__EU27__20261021_01_57_57.xlsx


## 6. Quick checks against the hand-built Sweden sheet

In [7]:
# Sweden KEEP sheet (area_comparison, 2024, BEFORE ipcc_rules.csv): WRI Settlements 4.07, Cropland 1.82, Forest 26.98, Grassland 7.81, Wetlands 0.75, Other 3.58 Mha
# After the rules most 'Other Land (Water)' is reassigned to Wetlands, so expect Wetlands up and Other down vs those numbers.
# CRT4.1 Sweden 2024: Settlements 1.97, Cropland 2.79, Forest (managed) 28.13, Grassland (managed) 0.51, Wetlands (managed) 0, Other 4.30 Mha
display(area_comparison[area_comparison.adm0 == "SWE"].round(3))

# Sweden KEEP sheet 'combined' (all gases, Mt CO2e/yr): LULUCF -46 reported by JRC; GNW veg Forest->Forest ~ -34.3
display(fluxes_CO2e[fluxes_CO2e.adm0 == "SWE"].round(2))
display(fluxes_CO2[fluxes_CO2.adm0 == "SWE"].round(2))

# EU27 totals
print("EU27 LULUCF, all gases (Mt CO2e/yr):")
display(fluxes_CO2e[fluxes_CO2e.category == "LULUCF"][["total"] + POOLS].sum().round(1))


,adm0,country_name,IPCC_class_name,WRI_Mha,CRT_Mha,difference_Mha,CRT_year,Notes
182,SWE,Sweden,Settlements,4.061,1.971,2.090,2024,
183,SWE,Sweden,Cropland,1.822,2.788,-0.966,2024,
184,SWE,Sweden,Forest land,26.959,28.126,-1.167,2024,
185,SWE,Sweden,Grassland,7.799,0.510,7.289,2024,
186,SWE,Sweden,Wetlands,4.015,0.000,4.015,2024,No managed Wetlands reported in CRT4.1
187,SWE,Sweden,Other,0.252,4.305,-4.053,2024,
188,SWE,Sweden,Total,44.907,37.699,7.208,2024,


,adm0,country_name,category,total,vegetation,mineral soils,organic soils
104,SWE,Sweden,LULUCF,207.67,-38.26,6.80,239.13
105,SWE,Sweden,FOREST,77.29,-25.17,-0.35,102.81
106,SWE,Sweden,DEFORESTATION,4.66,2.54,-0.00,2.12
107,SWE,Sweden,OTHER,125.72,-15.63,7.15,134.20


,adm0,country_name,category,total,vegetation,mineral soils,organic soils
104,SWE,Sweden,LULUCF,153.27,-38.27,6.80,184.74
105,SWE,Sweden,FOREST,53.56,-25.17,-0.35,79.09
106,SWE,Sweden,DEFORESTATION,4.18,2.54,-0.00,1.64
107,SWE,Sweden,OTHER,95.53,-15.63,7.15,104.01


EU27 LULUCF, all gases (Mt CO2e/yr):


total            264.0
vegetation      -486.6
mineral soils     83.9
organic soils    666.7
dtype: float64